# Credit Risk Prediction

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import shap
import joblib

In [3]:
# Load the dataset
df = pd.read_csv("./dataset/credit_risk_dataset.csv")
# Loan status (0 is non default 1 is default)

In [4]:
# print(df.shape)

# Display basic information about the dataset
# print(df.info())

# Display statistical summary
# print(df.describe())

# Display number of missing values
# print(df.isnull().sum())

# Display target distribution
# print(df["loan_status"].value_counts())

# Display target distribution as percentages
# print(df["loan_status"].value_counts(normalize=True) * 100)


In [6]:
# # -> First graph :- Default Distribution
# plt.figure(figsize=(6,5))
# sns.countplot(
#     data = df,
#     x = 'loan_status')
# plt.title("Loan Default Distribution")
# plt.xlabel("Loan Status")
# plt.ylabel("Number of Applications")

# plt.xticks([0,1], ["No Default", "Default"])
# plt.show()

# # -> Second graph:- Loan Grade vs Default

# plt.figure(figsize=(8,5))
# sns.countplot(data = df, x="loan_grade", hue="loan_status")

# plt.title("Loan Grade vs Default Status")
# plt.xlabel("Loan Grade")
# plt.ylabel("Number of Applications")

# plt.legend(["No Default","Default"], title="Loan Status")
# plt.show()

# # -> Third graph:- Income vs Default
# plt.figure(figsize=(8, 5))

# sns.boxplot(data=df,x="loan_status",y="person_income")

# plt.title("Income vs Loan Default")
# plt.xlabel("Loan Status")
# plt.ylabel("Annual Income")

# plt.xticks(
#     [0, 1],
#     ["No Default", "Default"]
# )

# plt.show()

# # -> Fourth graph:- Loan Interest Rate vs Default
# plt.figure(figsize=(8,5))

# sns.boxplot(data=df, x="loan_status", y="loan_int_rate")

# plt.title("Interest Rate vs Loan Default")
# plt.xlabel("Loan Status")
# plt.ylabel("Interest Rate (%)")

# plt.xticks([0,1], ["No Default", "Default"])
# plt.show()


# # -> Fifth graph:- Credit History vs Default
# plt.figure(figsize=(8,5))

# sns.boxplot(data = df, x="loan_status", y="cb_person_cred_hist_length")

# plt.title("Credit History Length vs Loan Default")
# plt.xlabel("Loan Status")
# plt.ylabel("Credit History Length (Years)")

# plt.xticks(
#     [0, 1],
#     ["No Default", "Default"]
# )

# plt.show()

In [10]:
# # Display unique values for categorical features
# print("Home Ownership:")
# print(df["person_home_ownership"].value_counts())

# print("\nLoan Intent:")
# print(df["loan_intent"].value_counts())

# print("\nLoan Grade:")
# print(df["loan_grade"].value_counts())

# print("\nPrevious Default:")
# print(df["cb_person_default_on_file"].value_counts())

In [11]:
# Separate input feature and target variable
X = df.drop(
    columns=["loan_status", "loan_grade"]
)

Y = df["loan_status"]
# print("Features shape:", X.shape)
# print("Target shape:", Y.shape)

In [12]:
# Define numerical features
numerical_features = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

# Define categorical features
categorical_features = [
    "person_home_ownership",
    "loan_intent",
   
    "cb_person_default_on_file"
]


In [13]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=42, stratify=Y)

# Stratify = It ensures the training and testing datasets maintain approximately the same default/non-default ratio.

In [14]:
# Numerical Preprocessing
numerical_transformer = Pipeline(steps=[
     ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Replace missing value -> with median -> StandardScaler

In [15]:
# Categorical preprocessing
categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [16]:
# Combine numerical and Categorical Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Now we have preprocessed X

In [17]:
# logistic Regression Pipeline
logistic_model = Pipeline(

    steps = [
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

logistic_model.fit(X_train, Y_train)
# Prediction on test dataset
Y_pred = logistic_model.predict(X_test)

# Get probability of the positive class
Y_probability = logistic_model.predict_proba(X_test)[:,1]

In [18]:
# Calculating evaluation metrics
accuracy = accuracy_score(Y_test, Y_pred)
precision = precision_score(Y_test, Y_pred)
recall = recall_score(Y_test, Y_pred)
f1 = f1_score(Y_test, Y_pred)
roc_auc = roc_auc_score(Y_test, Y_probability)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

# Display detailed classification report
print("\nClassification Report (Logistic Regression Model):")
print(classification_report(Y_test, Y_pred))

# Generate confusion matrix
cm = confusion_matrix(Y_test, Y_pred)

print("\nConfusion Matrix:")
print(cm)

Accuracy : 0.8507
Precision: 0.7470
Recall   : 0.4775
F1 Score : 0.5826
ROC-AUC  : 0.8547

Classification Report (Logistic Regression Model):
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      5095
           1       0.75      0.48      0.58      1422

    accuracy                           0.85      6517
   macro avg       0.81      0.72      0.75      6517
weighted avg       0.84      0.85      0.84      6517


Confusion Matrix:
[[4865  230]
 [ 743  679]]


In [20]:
# # Visualize confusion matrix
# plt.figure(figsize=(6,5))

# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Default", "Default"],
#     yticklabels=["No Default", "Default"] )

# plt.title("Confusion Matrix - Logistic Regression")
# plt.xlabel("Predicted")
# plt.ylabel("Actual")

# plt.show()


# #ROC Curve

# fpr, tpr, thresholds = roc_curve(
#     Y_test,
#     Y_probability
# )

# # Plot ROC curve
# plt.figure(figsize=(7, 5))

# plt.plot(
#     fpr,
#     tpr,
#     label=f"Logistic Regression (AUC = {roc_auc:.3f})"
# )

# plt.plot(
#     [0, 1],
#     [0, 1],
#     linestyle="--",
#     label="Random Classifier"
# )

# plt.title("ROC Curve - Logistic Regression")
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.legend()
# plt.show()

In [108]:
# Decision Tree Pipeline
decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(random_state=42))
    ]
)

# Train the model
decision_tree_model.fit(X_train, Y_train)

# prediction
Y_pred_dt = decision_tree_model.predict(X_test)

# Get probability of default class
Y_probability_dt = decision_tree_model.predict_proba(X_test)[:,1]

In [130]:
# Caculating evaulation metrics
accuracy_dt = accuracy_score(Y_test, Y_pred_dt)
precision_dt = precision_score(Y_test, Y_pred_dt)
recall_dt = recall_score(Y_test, Y_pred_dt)
f1_dt = f1_score(Y_test, Y_pred_dt)
roc_auc_dt = roc_auc_score(Y_test, Y_probability_dt)
print(f"Accuracy : {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall   : {recall_dt:.4f}")
print(f"F1 Score : {f1_dt:.4f}")
print(f"ROC-AUC  : {roc_auc_dt:.4f}")

print("\nClassification Report (Decision Tree Model):")
print(classification_report(Y_test, Y_pred_dt))

# Confusion Matrix
cm_dt = confusion_matrix(Y_test, Y_pred_dt)
print("\nConfusion Matrix:")
print(cm_dt)

Accuracy : 0.8782
Precision: 0.7096
Recall   : 0.7475
F1 Score : 0.7281
ROC-AUC  : 0.8311

Classification Report (Decision Tree Model):
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      5095
           1       0.71      0.75      0.73      1422

    accuracy                           0.88      6517
   macro avg       0.82      0.83      0.82      6517
weighted avg       0.88      0.88      0.88      6517


Confusion Matrix:
[[4660  435]
 [ 359 1063]]


In [21]:
# # Visualize Decision Tree Confusion matrix
# plt.figure(figsize=(6, 5))

# sns.heatmap(
#     cm_dt,
#     annot=True,
#     fmt="d",
#     cmap="Blues",
#     xticklabels=["No Default", "Default"],
#     yticklabels=["No Default", "Default"]
# )

# plt.title("Confusion Matrix - Decision Tree")
# plt.xlabel("Predicted")
# plt.ylabel("Actual")

# plt.show()

# # Calculate ROC curve
# fpr_dt, tpr_dt, thresholds_dt = roc_curve(
#     Y_test,
#     Y_probability_dt
# )

# # Plot ROC curve
# plt.figure(figsize=(7, 5))

# plt.plot(
#     fpr_dt,
#     tpr_dt,
#     label=f"Decision Tree (AUC = {roc_auc_dt:.3f})"
# )

# plt.plot(
#     [0, 1],
#     [0, 1],
#     linestyle="--",
#     label="Random Classifier"
# )

# plt.title("ROC Curve - Decision Tree")
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.legend()

# plt.show()

In [22]:
# Random Forest Pipeline
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train the model
random_forest_model.fit(X_train, Y_train)

# Prediction
Y_pred_rf = random_forest_model.predict(X_test)

# Probability of default
Y_probability_rf = random_forest_model.predict_proba(X_test)[:,1]

In [23]:
# Caculating evaulation metrics
accuracy_rf = accuracy_score(Y_test, Y_pred_rf)
precision_rf = precision_score(Y_test, Y_pred_rf)
recall_rf = recall_score(Y_test, Y_pred_rf)
f1_rf = f1_score(Y_test, Y_pred_rf)
roc_auc_rf = roc_auc_score(Y_test, Y_probability_rf)

print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")
print(f"ROC-AUC  : {roc_auc_rf:.4f}")

print("\nClassification Report (Random Forest):")
print(classification_report(Y_test, Y_pred_rf))

# Generate confusion matrix
cm_rf = confusion_matrix(Y_test, Y_pred_rf)

print("\nConfusion Matrix:")
print(cm_rf)

Accuracy : 0.9211
Precision: 0.9522
Recall   : 0.6723
F1 Score : 0.7881
ROC-AUC  : 0.9249

Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.92      0.99      0.95      5095
           1       0.95      0.67      0.79      1422

    accuracy                           0.92      6517
   macro avg       0.93      0.83      0.87      6517
weighted avg       0.92      0.92      0.92      6517


Confusion Matrix:
[[5047   48]
 [ 466  956]]


In [24]:
# plt.figure(figsize=(6, 5))

# sns.heatmap(
#     cm_rf,
#     annot=True,
#     fmt="d",
#     xticklabels=["No Default", "Default"],
#     yticklabels=["No Default", "Default"]
# )

# plt.title("Confusion Matrix - Random Forest")
# plt.xlabel("Predicted")
# plt.ylabel("Actual")

# plt.show()

In [25]:
# Balance Random Forest

balance_rf_model = Pipeline(
    
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train the Balanced Random Forest
balance_rf_model.fit(X_train, Y_train)

# Prediction
Y_pred_balanced_rf = balance_rf_model.predict(X_test)

# Get probability of default
Y_probability_balanced_rf = (
    balance_rf_model.predict_proba(X_test)[:,1]
)


In [26]:
# Calculating evaluation matrix
accuracy_balanced_rf = accuracy_score(Y_test, Y_pred_balanced_rf)
precision_balanced_rf = precision_score(Y_test, Y_pred_balanced_rf)
recall_balanced_rf = recall_score(Y_test, Y_pred_balanced_rf)
f1_balanced_rf = f1_score(Y_test, Y_pred_balanced_rf)
roc_auc_balanced_rf = roc_auc_score(Y_test, Y_pred_balanced_rf)

print(f"Accuracy : {accuracy_balanced_rf:.4f}")
print(f"Precision: {precision_balanced_rf:.4f}")
print(f"Recall   : {recall_balanced_rf:.4f}")
print(f"F1 Score : {f1_balanced_rf:.4f}")
print(f"ROC-AUC  : {roc_auc_balanced_rf:.4f}")
print("\nClassification Report (Balance Random Forest Model):")
print(classification_report(Y_test, Y_pred_balanced_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred_balanced_rf))

Accuracy : 0.9205
Precision: 0.9556
Recall   : 0.6667
F1 Score : 0.7854
ROC-AUC  : 0.8290

Classification Report (Balance Random Forest Model):
              precision    recall  f1-score   support

           0       0.91      0.99      0.95      5095
           1       0.96      0.67      0.79      1422

    accuracy                           0.92      6517
   macro avg       0.93      0.83      0.87      6517
weighted avg       0.92      0.92      0.92      6517


Confusion Matrix:
[[5051   44]
 [ 474  948]]


In [27]:
# Fitted preprocessing Pipeline
fitted_preprocessor = random_forest_model.named_steps["preprocessor"]

# get trained Random Forest Model
fitted_rf = random_forest_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

# print("Number of features:", len(feature_names))
# print(feature_names)

# Get feature importance from Random Forest
importances = fitted_rf.feature_importances_
# print(importances)
# Create DataFrame containing feature names and importance
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Display the top 15 most important features
print(feature_importance_df.head(15))

                                Feature  Importance
5              num__loan_percent_income    0.236129
4                    num__loan_int_rate    0.181094
1                    num__person_income    0.156284
3                        num__loan_amnt    0.083302
2                num__person_emp_length    0.059669
0                       num__person_age    0.051958
10      cat__person_home_ownership_RENT    0.049951
6       num__cb_person_cred_hist_length    0.040817
7   cat__person_home_ownership_MORTGAGE    0.029606
9        cat__person_home_ownership_OWN    0.018442
11   cat__loan_intent_DEBTCONSOLIDATION    0.016677
14             cat__loan_intent_MEDICAL    0.015533
13     cat__loan_intent_HOMEIMPROVEMENT    0.013974
17     cat__cb_person_default_on_file_N    0.010517
18     cat__cb_person_default_on_file_Y    0.010146


In [29]:
# # Create a feature importance graph
# # Select top 15 features
# top_features = feature_importance_df.head(15)

# # Plot feature importance
# plt.figure(figsize=(10, 7))

# sns.barplot(
#     data=top_features,
#     x="Importance",
#     y="Feature"
# )

# plt.title("Top 15 Feature Importances - Random Forest")
# plt.xlabel("Importance")
# plt.ylabel("Feature")

# plt.tight_layout()
# plt.show()

In [30]:
# XGBoost Pipeline
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

# Train XGBoost model
xgb_model.fit(X_train, Y_train)

# Prediction
Y_pred_xgb= xgb_model.predict(X_test)
# Get probability of default
Y_probability_xgb = xgb_model.predict_proba(X_test)[:,1]



In [31]:
# Calculating XGBoost metrics
accuracy_xgb = accuracy_score(Y_test, Y_pred_xgb)
precision_xgb = precision_score(Y_test, Y_pred_xgb)
recall_xgb = recall_score(Y_test, Y_pred_xgb)
f1_xgb = f1_score(Y_test,Y_pred_xgb)
roc_auc_xgb = roc_auc_score(Y_test, Y_probability_xgb)

print(f"Accuracy : {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall   : {recall_xgb:.4f}")
print(f"F1 Score : {f1_xgb:.4f}")
print(f"ROC-AUC  : {roc_auc_xgb:.4f}")

print("\nClassification Report (XGBoost Model):")
print(classification_report(Y_test, Y_pred_xgb))

cm_xgb = confusion_matrix(Y_test, Y_pred_xgb)
print("\nConfusion Matrix:")
print(cm_xgb)

Accuracy : 0.9257
Precision: 0.9699
Recall   : 0.6807
F1 Score : 0.8000
ROC-AUC  : 0.9419

Classification Report (XGBoost Model):
              precision    recall  f1-score   support

           0       0.92      0.99      0.95      5095
           1       0.97      0.68      0.80      1422

    accuracy                           0.93      6517
   macro avg       0.94      0.84      0.88      6517
weighted avg       0.93      0.93      0.92      6517


Confusion Matrix:
[[5065   30]
 [ 454  968]]


In [33]:
# # Visualize Confusion Matrix
# plt.figure(figsize=(6, 5))

# sns.heatmap(
#     cm_xgb,
#     annot=True,
#     fmt="d",
#     cmap="Blues",
#     xticklabels=["No Default", "Default"],
#     yticklabels=["No Default", "Default"]
# )

# plt.title("Confusion Matrix - XGBoost")
# plt.xlabel("Predicted")
# plt.ylabel("Actual")

# plt.tight_layout()
# plt.show()

In [34]:

param_grid = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

In [35]:
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions= param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs= -1
)
random_search.fit(X_train, Y_train)
print("Best Parameters:")
print(random_search.best_params_)
print("Best Cross-Validation ROC-AUC:")
print(random_search.best_score_)
best_xgb_model = random_search.best_estimator_
Y_pred_xgb_tuned = best_xgb_model.predict(X_test)
Y_probability_xgb_tuned = (
    best_xgb_model.predict_proba(X_test)[:,1]
)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Parameters:
{'model__subsample': 1.0, 'model__n_estimators': 200, 'model__min_child_weight': 5, 'model__max_depth': 5, 'model__learning_rate': 0.2, 'model__colsample_bytree': 1.0}
Best Cross-Validation ROC-AUC:
0.9374451857078366


In [36]:
accuracy_xgb_tuned = accuracy_score(Y_test, Y_pred_xgb_tuned)
precision_xgb_tuned = precision_score(Y_test, Y_pred_xgb_tuned)
recall_xgb_tuned = recall_score(Y_test, Y_pred_xgb_tuned)
f1_xgb_tuned = f1_score(Y_test, Y_pred_xgb_tuned)
roc_auc_xgb_tuned = roc_auc_score(Y_test, Y_probability_xgb_tuned)

print(f"Accuracy : {accuracy_xgb_tuned:.4f}")
print(f"Precision: {precision_xgb_tuned:.4f}")
print(f"Recall   : {recall_xgb_tuned:.4f}")
print(f"F1 Score : {f1_xgb_tuned:.4f}")
print(f"ROC-AUC  : {roc_auc_xgb_tuned:.4f}")

print("\nClassification Report (XGB Tuned):")
print(classification_report(Y_test, Y_pred_xgb_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred_xgb_tuned))

Accuracy : 0.9271
Precision: 0.9446
Recall   : 0.7075
F1 Score : 0.8090
ROC-AUC  : 0.9481

Classification Report (XGB Tuned):
              precision    recall  f1-score   support

           0       0.92      0.99      0.95      5095
           1       0.94      0.71      0.81      1422

    accuracy                           0.93      6517
   macro avg       0.93      0.85      0.88      6517
weighted avg       0.93      0.93      0.92      6517


Confusion Matrix:
[[5036   59]
 [ 416 1006]]


In [37]:


# | Model                  |   Accuracy |  Precision |     Recall |         F1 |    ROC-AUC |
# | ---------------------- | ---------: | ---------: | ---------: | ---------: | ---------: |
# | Logistic Regression    |     85.07% |     74.70% |     47.75% |     58.26% |     85.47% |
# | Decision Tree          |     87.82% |     70.96% |     74.75% |     72.81% |     83.11% |
# | Random Forest          |     92.11% |     95.22% |     67.23% |     78.81% |     92.49% |
# | Balanced Random Forest |     92.05% |     95.56% |     66.67% |     78.54% |     82.90% |
# | XGBoost                |     92.57% |     96.99% |     68.07% |     80.00% |     94.19% |
# | **Tuned XGBoost**      | **92.71%** | **94.46%** | **70.75%** | **80.90%** | **94.81%** |


In [38]:
# get fitted preprocessor
fitted_preprocessor = best_xgb_model.named_steps["preprocessor"]

# Get fitted XGBoost model
fitted_xgb = best_xgb_model.named_steps["model"]

# Transform our test data
X_test_transformed = fitted_preprocessor.transform(X_test)

# Get feature name
feature_names = fitted_preprocessor.get_feature_names_out()
# print(feature_name)

explainer = shap.TreeExplainer(fitted_xgb)
shap_values = explainer(X_test_transformed)

# shap.summary_plot(
#     shap_values, X_test_transformed, feature_names = feature_names
# )

In [41]:
shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": np.abs(shap_values.values).mean(axis=0)
})
shap_importance = shap_importance.sort_values(
    by = "Importance", ascending=False
)
print(shap_importance.head(15))
# plt.figure(figsize=(10, 6))

# shap_importance.head(15).sort_values(
#     by="Importance"
# ).plot(
#     x="Feature",
#     y="Importance",
#     kind="barh",
#     legend=False
# )

# plt.title("Top 15 Features Affecting Credit Risk")
# plt.xlabel("Mean Absolute SHAP Value")
# plt.ylabel("Feature")

# plt.tight_layout()
# plt.show()
# shap.summary_plot(
#     shap_values,
#     X_test_transformed,
#     feature_names=feature_names
# )
# shap.summary_plot(
#     shap_values,
#     X_test_transformed,
#     feature_names=feature_names,
#     plot_type="bar"
# )



                                Feature  Importance
1                    num__person_income    1.039328
4                    num__loan_int_rate    0.930328
5              num__loan_percent_income    0.824374
10      cat__person_home_ownership_RENT    0.448411
9        cat__person_home_ownership_OWN    0.376740
3                        num__loan_amnt    0.307325
16             cat__loan_intent_VENTURE    0.290541
13     cat__loan_intent_HOMEIMPROVEMENT    0.191495
11   cat__loan_intent_DEBTCONSOLIDATION    0.165254
2                num__person_emp_length    0.163728
0                       num__person_age    0.130534
17     cat__cb_person_default_on_file_N    0.129931
14             cat__loan_intent_MEDICAL    0.125477
7   cat__person_home_ownership_MORTGAGE    0.110765
6       num__cb_person_cred_hist_length    0.069277


In [128]:
joblib.dump(
    best_xgb_model,
    "models/credit_risk_xgboost.pkl"
)

['models/credit_risk_xgboost.pkl']